In [ ]:
# ======================
# CONFIGURATION DU CHEMIN DU PROJET
# ======================
import sys
import os

# Remonter à la racine du projet (depuis notebooks/training/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Racine du projet ajoutée au PYTHONPATH : {project_root}")
print(f"   sys.path : {sys.path[0]}")

✅ Racine du projet ajoutée au PYTHONPATH : c:\Users\SOL\Downloads\sentiment_analysis_project
   sys.path : c:\Users\SOL\Downloads\sentiment_analysis_project


In [ ]:

# 2. Imports
from src.models import SentimentTransformer
from src.trainer import train_model
from src.dataset import create_dataloaders
from src.preprocessing import preprocess_full_dataset
import torch
import torch.nn as nn

# 3. Prétraitement + Dataloaders (sans lemmatisation !)
df_clean, _ = preprocess_full_dataset(
    r'C:\Users\SOL\Downloads\sentiment_analysis_project\data\sentiment_dataset.csv'
)
train_loader, test_loader, vocab, train_df, test_df = create_dataloaders(
    df_clean,
    max_length=60,
    batch_size=64
)

# 4. Initialisation du modèle
model = SentimentTransformer(
    vocab_size=len(vocab),
    embed_dim=256,
    num_heads=8,
    mlp_dim=512,
    transformer_layers=4,
    max_length=60,
    dropout=0.3,
    num_classes=3
).to('cuda' if torch.cuda.is_available() else 'cpu')

print(f"🧠 SentimentTransformer initialisé :")
print(f"   - Embedding dim   : 256")
print(f"   - Têtes d'att.    : 8")
print(f"   - Couches         : 4")
print(f"   - MLP dim         : 512")
print(f"   - Paramètres      : {sum(p.numel() for p in model.parameters()):,}")




✅ Ressource NLTK 'punkt' déjà présente
📥 Téléchargement de la ressource NLTK 'wordnet'...
✅ 'wordnet' téléchargé avec succès
✅ Ressource NLTK 'stopwords' déjà présente
📥 Téléchargement de la ressource NLTK 'omw-1.4'...
✅ 'omw-1.4' téléchargé avec succès
🚀 PRÉTRAITEMENT DU DATASET

✅ Dataset chargé : 31,232 échantillons
🧹 Prétraitement en cours...
✅ 31,202 échantillons conservés

📏 Longueur recommandée (95e percentile) : 56 tokens

📚 Vocabulaire construit : 11703 mots uniques

📦 Dataloaders créés :
   - Train : 391 batches
   - Test  : 98 batches
🧠 SentimentTransformer initialisé :
   - Embedding dim   : 256
   - Têtes d'att.    : 8
   - Couches         : 4
   - MLP dim         : 512
   - Paramètres      : 5,121,539


In [ ]:
# 5. Entraînement
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)  # AdamW + weight decay

history, model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    num_epochs=25,
    patience=5,
    save_dir=r'C:\Users\SOL\Downloads\sentiment_analysis_project\models\trainary',
    model_name='sentiment_transformer',
    verbose=True
)




🚀 DÉBUT DE L'ENTRAÎNEMENT : sentiment_transformer
Epoch  | Train Loss   Train F1   Val Loss     Val F1     Val Acc    | Time   | Status      
--------------------------------------------------------------------------------
     1 | 1.1573       0.3270     1.1044       0.1675     0.3355     | 11    s | ★ BEST      
     2 | 1.0677       0.3819     0.8981       0.5033     0.5494     | 10    s | ★ BEST      
     3 | 0.9070       0.5564     0.8152       0.6371     0.6382     | 10    s | ★ BEST      
     4 | 0.8271       0.6166     0.7777       0.6531     0.6582     | 10    s | ★ BEST      
     5 | 0.7707       0.6543     0.7575       0.6777     0.6775     | 10    s | ★ BEST      
     6 | 0.7317       0.6803     0.7570       0.6677     0.6738     | 10    s | →           
     7 | 0.6956       0.7015     0.7358       0.6887     0.6861     | 10    s | ★ BEST      
     8 | 0.6743       0.7132     0.7503       0.6842     0.6866     | 10    s | →           
     9 | 0.6474       0.7285    

In [ ]:
# 6. Évaluation
from src.evaluate import evaluate_model

results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    class_names=['Négatif', 'Neutre', 'Positif'],
    save_dir=r'C:\Users\SOL\Downloads\sentiment_analysis_project\reports\figures\sentiment_transformer',
    model_name='sentiment_transformer')


📊 ÉVALUATION DU MODÈLE : sentiment_transformer

Accuracy      : 0.6904
F1 Macro      : 0.6904
F1 Weighted   : 0.6885
ROC AUC Micro : 0.8536
ROC AUC Macro : 0.8509

Classification Report:
              precision    recall  f1-score   support

     Négatif     0.6928    0.6615    0.6768      1820
      Neutre     0.6384    0.6085    0.6231      2327
     Positif     0.7392    0.8066    0.7714      2094

    accuracy                         0.6904      6241
   macro avg     0.6901    0.6922    0.6904      6241
weighted avg     0.6881    0.6904    0.6885      6241

   → Matrice de confusion sauvegardée : confusion_matrix.png/csv
   → Courbes ROC sauvegardées : roc_curves.png

✅ Évaluation terminée - Résultats sauvegardés dans : C:\Users\SOL\Downloads\sentiment_analysis_project\reports\figures\sentiment_transformer\sentiment_transformer
